# Stage 5: Phase 2 Packaging & Dynamic Champion Verification

This notebook prepares the submission archive for Phase 2 verification. It dynamically detects the winning champion architecture selected by `07_model_comparison.ipynb` (either **BanglaT5** or **TituLLM-3B**), verifies compliance with the $\le 3.0\text{B}$ parameter cap, runs a demo inference, generates the appropriate `model_card.md`, and outputs the complete reproducibility package.

### 1. Load Champion Verdict and Inspect Model Parameters

In [ ]:
import os
import sys
import json
import torch
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer

sys.path.append(os.path.abspath('src'))
import inference_utils

verdict_path = "/kaggle/working/champion_model.json" if os.path.exists("/kaggle/working/champion_model.json") else "champion_model.json"

if os.path.exists(verdict_path):
    with open(verdict_path, "r", encoding="utf-8") as f:
        verdict = json.load(f)
    champion_type = verdict.get("champion", "banglat5")
    print(f"Loaded champion verdict from {verdict_path}: {champion_type.upper()}")
    print(f"Validation score: {verdict.get('champion_val_score', 'N/A')}")
else:
    # Default detection based on available model directories
    if os.path.exists("/kaggle/working/banglat5_final") or os.path.exists("banglat5_final"):
        champion_type = "banglat5"
    else:
        champion_type = "titulm-3b"
    print(f"No verdict file found. Defaulting to champion: {champion_type}")

if champion_type == "banglat5":
    model_dir = "/kaggle/working/banglat5_final" if os.path.exists("/kaggle/working/banglat5_final") else "banglat5_final"
    print(f"Loading champion model ({champion_type}) from: {model_dir}...")
    model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=False)
else:
    model_dir = "/kaggle/working/final_model" if os.path.exists("/kaggle/working/final_model") else "final_model"
    print(f"Loading champion model ({champion_type}) from: {model_dir}...")
    model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=torch.bfloat16, device_map="cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)

param_count = sum(p.numel() for p in model.parameters())
print(f"\n=== Parameter Count Verification for Champion ({champion_type}) ===")
print(f"Exact Parameter Count: {param_count:,}")

if param_count <= 3_000_000_000:
    margin = 3_000_000_000 - param_count
    print(f"PASS: {param_count:,} parameters is strictly below the 3,000,000,000 parameter cap.")
    print(f"Safety Margin: {margin:,} parameters below the competition ceiling ({margin / 3e9 * 100:.1f}% headroom).")
else:
    raise ValueError(f"FAIL: Model exceeds 3B limit: {param_count:,}")

### 2. Demo Inference Over 5 Sample Medical Prompts

In [ ]:
demo_prompts = [
    "আমার কিছুদিন ধরে মাথা ব্যথা হচ্ছে এবং বমি বমি ভাব হচ্ছে। আমার কী করা উচিত?",
    "বাচ্চার খুব বেশি মাত্রায় জ্বর এবং সর্দি আছে। ডক্টরের কাছে কখন নেওয়া উচিত?",
    "খাবারের পর পেটে গ্যাস হচ্ছে ও জ্বালাপোড়া করে। প্রতিকার কী?",
    "হঠাৎ করে বুকে চাপ ধরা ব্যথা অনুভব করছি। এটা কি স্ট্রোক হতে পারে?",
    "অতিরিক্ত চুল পড়ার সমাধান কীভাবে পেতে পারি?"
]

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print(f"=== Running Sample Demo Inference with Champion Model ({champion_type}) ===\n")

for i, prompt in enumerate(demo_prompts, 1):
    if champion_type == "banglat5":
        cleaned = inference_utils.generate_banglat5(
            model=model,
            tokenizer=tokenizer,
            patient_input=prompt,
            num_beams=5,
            no_repeat_ngram_size=3,
            length_penalty=0.6,
            max_new_tokens=300,
            min_new_tokens=40
        )
    else:
        candidates = inference_utils.generate_candidates(model, tokenizer, prompt, k=4)
        selected = inference_utils.mbr_select(candidates)
        cleaned = inference_utils.clean_output(selected)
        
    print(f"Sample {i}:")
    print("Input: ", prompt)
    print("Output:", cleaned)
    print("-" * 80)

### 3. Generate Model Card Documentation

In [ ]:
if champion_type == "banglat5":
    model_card_content = f"""# Model Card — Bengali Medical Dialogue Model (BanglaT5 Full Fine-Tune)

## Model Metadata
- **Architecture**: `csebuetnlp/banglat5` (T5-base Sequence-to-Sequence Encoder-Decoder)
- **Total Parameters**: {param_count:,} (~247 Million parameters, well under the 3.0B parameter limit with >90% headroom)
- **Fine-Tuning Strategy**: End-to-end full parameter fine-tuning (not LoRA) on target Bengali medical dialogues
- **Training Hyperparameters**:
  - Optimizer: AdamW (`learning_rate = 3e-4`, `weight_decay = 1e-6`)
  - Epochs: 6
  - Warmup Ratio: 0.1
  - Label Smoothing: 0.1
  - Mixed Precision: bfloat16 / fp16
  - Checkpoint Selection: Best validation loss checkpoint (`eval_loss`)

## Training Data
- **Dataset**: Cleaned official competition training split (`sft_train.csv`) preprocessed with Bengali text normalization (`csebuetnlp/normalizer`).

## Inference Specifications
- **Normalization**: `normalizer.normalize()` applied to input strings prior to tokenization
- **Decoding Strategy**: Deterministic Beam Search (`num_beams=5`, `no_repeat_ngram_size=3`, `length_penalty=0.6`)
- **Token Limits**: `max_new_tokens = 300`, `min_new_tokens = 40`
"""
else:
    model_card_content = f"""# Model Card — Bengali Medical Dialogue Model (TituLLM-3B QLoRA)

## Model Metadata
- **Base Model**: `hishab/titulm-llama-3.2-3b-v2.0`
- **Total Parameters**: {param_count:,} (~3.2B base architecture merged)
- **Fine-Tuning Strategy**: Two-stage QLoRA fine-tuning
  - LoRA Rank (r): 32, Alpha: 64, Dropout: 0.05
  - Learning Rate: Stage 1 = 2e-4, Stage 2 = 5e-5 (Cosine scheduler with warmup)
  - Target Modules: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`

## Training Data
- **Stage 1**: Official `train.csv` + normalized `shetumohanto/doctor_qa_bangla`
- **Stage 2**: Cleaned official `sft_train.csv` anchor data

## Inference Specifications
- **Decoding**: $k=4$ candidate sampling (temperature=0.7, top_p=0.9)
- **Selection**: Minimum Bayes Risk (MBR) selection over pairwise ROUGE-L with length filtering [65, 145]
"""

model_card_path = os.path.join(model_dir, "model_card.md")
with open(model_card_path, "w", encoding="utf-8") as f:
    f.write(model_card_content)

print(f"Saved updated model_card.md to: {model_card_path}")

### 4. Create Final Phase 2 Reproducibility Package Zip

In [ ]:
import shutil

zip_base_path = "/kaggle/working/phase2_submission"

# Archive model folder
target_model_folder = os.path.basename(model_dir)
parent_dir = os.path.dirname(model_dir) if os.path.dirname(model_dir) else "."

print(f"Creating Phase 2 submission zip for '{target_model_folder}'...")
shutil.make_archive(zip_base_path, 'zip', root_dir=parent_dir, base_dir=target_model_folder)

zip_file = f"{zip_base_path}.zip"
size_mb = os.path.getsize(zip_file) / (1024 * 1024)
print(f"Submission Package zipped: {zip_file}")
print(f"Archive Size: {size_mb:.2f} MB")

### 5. Final Reproducibility Checklist & File Audit

In [ ]:
print("=========================================================================================")
print("                       PHASE 2 PACKAGING COMPLETE AUDIT")
print("=========================================================================================")
print(f"Champion Model:                {champion_type.upper()}")
print(f"Packaged Model Directory:      {model_dir}")
print(f"Parameter Limit Check:         {param_count:,} (PASS, < 3,000,000,000 cap)")
print(f"Submission Zip File:           {zip_file} ({size_mb:.2f} MB)")

print("\n--- Included Pipeline & Evaluation Notebooks ---")
notebooks = [
    "01_data_pipeline.ipynb",
    "02_train_qlora.ipynb",
    "02b_train_banglat5.ipynb",
    "03_inference_and_submit.ipynb",
    "03b_inference_banglat5.ipynb",
    "04_local_validation.ipynb",
    "05_phase2_packaging.ipynb",
    "06_retrieval_addon.ipynb",
    "07_model_comparison.ipynb"
]
for nb in notebooks:
    status = "EXISTS (READY)" if os.path.exists(nb) or os.path.exists(f"/kaggle/working/{nb}") else "OPTIONAL"
    print(f"  [OK] {nb:<30} -> {status}")

print("=========================================================================================")